# 截断正态分布的拒绝抽样

> 对应代码：`MathStatsCode/code_in_notes/Chap.5/rejection_trunc.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：以指数分布为工具分布，对标准正态分布在 x>3 上的截断分布做拒绝抽样，并画图展示接受/拒绝的样本点。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.5`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.5")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

清空数据、关闭分页、设定种子并准备1000个候选观测。本节要用拒绝抽样从**标准正态在 x>3 上的截断分布**（即条件分布 N(0,1)|x>3）中抽样。


In [ ]:
%%stata
clear
set more off
set seed 505
set obs 1000


**参数**

`scalar c=3`：用 `scalar` 定义标量常数 c=3，即截断点。标量与 local 宏不同，它可以在命令之间持久存在（本节大量使用标量存中间结果）。


In [ ]:
%%stata
scalar c=3


**计算lambda和M**

这一段计算**目标密度的归一化常数**并设定**工具分布与包络常数**（对应代码中的四行 scalar）：
- `temp_cons=sqrt(2*pi)*(1-normal(c))`：截断分布的密度为 f(x)=φ(x)/(1-Φ(c))（φ 为标准正态密度 `normalden`、Φ 为分布函数 `normal`），这里先把 √(2π)·(1-Φ(3)) 存下来备用；
- 选取工具分布为**平移的指数分布** g(x)=λe^{-λ(x-c)}（x≥c），先取 λ=2.5。拒绝抽样要求包络 M·g(x) 处处不小于 f(x)，程序中把 M 取为 f/g 的最大值：f/g ∝ exp(-x²/2+λx-λc)/λ，其最大值由 -x²/2+λx 的峰值（在 x=λ 处）得到，故 `M=exp(λ²/2-λc-log λ)/temp_cons`。
- 注意这里 λ=2.5<c=3，f/g 在 x≥c 上实际单调下降、最大值在 x=c 处取得，因此该 M 是一个**合法但不紧的上界**——包络线偏“松”、拒绝率偏高。下一节的最优 λ 将修正这一点。


In [ ]:
%%stata
scalar pi=3.1415926
scalar temp_cons=sqrt(2*pi)*(1-normal(c))
// scalar lam=(c+sqrt(c^2+4))/2
scalar lam=2.5
scalar M=(exp(lam^2/2-lam*c-log(lam)))/temp_cons


**产生均匀分布和指数分布**

用**逆变换法**从指数工具分布抽样：若 U~U(0,1)，则 x=c-ln(U)/λ 服从平移指数分布 g。`gen u=runiform()` 生成接受-拒绝比较所需的均匀纵坐标。


In [ ]:
%%stata
gen x=c-log(runiform())/lam
gen u=runiform()


**截断正态分布密度函数**

`gen density=normalden(x)/(1-normal(c))`：计算每个候选点处**目标密度** f(x) 的值（截断正态密度）；`label variable` 给变量加标签，便于画图图例显示。


In [ ]:
%%stata
gen density=normalden(x)/(1-normal(c))
label variable density "f(x)"


**指数分布密度函数（用M标准化，即 M*g(x)）**

`gen density_exp=M*lam*exp(-1*lam*(x-c))`：计算**包络曲线** M·g(x) 的值；同样加标签。拒绝抽样中包络曲线必须处处位于目标密度之上——后面画图可以直观看到这一点（λ=2.5 时包络线较“松”）。


In [ ]:
%%stata
gen density_exp=M*lam*exp(-1*lam*(x-c))
label variable density_exp "M*g(x)"


**挑出使得u<l/(M*g)的x，**

`gen selected=u<=(density/density_exp)`：接受判据。由于纵坐标 u~U(0,1) 乘以 density_exp 后落在 [0,M·g(x)]，故等价于“u ≤ f(x)/(M·g(x)) 则接受”。selected=1 表示接受、0 表示拒绝。


In [ ]:
%%stata
gen selected=u<=(density/density_exp)


**画图**

`sort x`：按 x 排序，使下一步折线图的连线按 x 从小到大连接。


In [ ]:
%%stata
sort x


**显示方便，散点图y轴用标准化的密度函数值乘以u**

这一段是**可视化**：先 `gen u_y=u*density_exp` 把均匀纵坐标换算到密度尺度（便于把接受/拒绝的点画在密度曲线之间）。`twoway` 叠加四个图层——虚线为包络 M·g(x)、黑实线为目标密度 f(x)、叉号为拒绝点、对号为接受点；`legend` 自定义图例文字、`graphr()` 设白色背景。最后一幅图清晰展示了拒绝抽样的几何原理：接受点是落在目标密度曲线下方（包络线之内）的点。


In [ ]:
%%stata
gen u_y=u*density_exp
twoway (line density_exp x, lpattern(dash)) ///
     (line density x, lc(black)) ///
     (scatter u_y x if selected==0, msize(small) msymbol(x) mc(gs8)) ///
     (scatter u_y x if selected==1, msize(small) msymbol(v) mc(gs10)) ///
     , legend(on ring(0) pos(1)  ///
     label(3 "Rejected") label(4 "Accepted") ///
     label(2 "Density of Accepted")) ///
     graphr(fcolor(white) color(white))
